# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [ ]:
import sys, os

# Add LASR-main project root so `src.*` imports work.
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
from src.configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig, DatasetConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUSAE

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/workspace/LASR-main/.venv/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2044, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/LASR-main/.venv/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2238, in _get_module
    raise e
  File "/workspace/LASR-main/.venv/lib/python3.12/site-packages/transformers/utils/import_utils.py", line 2236, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in

# Configuration

In [ ]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT_TAGS
use_few_shot = False

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=40,
    width="65k",
    l0="medium",
)

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

# Setup — HF Token

In [ ]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [ ]:
esnli_dataset = ESNLI_Dataset(dataset_config)
esnli_df = esnli_dataset.get_sample_dataframe(n=len(esnli_dataset))
esnli_df.head()

# Build Prompts

In [ ]:
prompted_data = esnli_dataset.build_prompts()
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])

# Load Model + SAE

In [ ]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer
sae = JumpReLUSAE.from_pretrained(sae_config, device=model_config.device)

## Generate and Gather Activations

In [ ]:
import textwrap

# Pick one sample prompt and generate
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()

generation, full_ids, prompt_len = model.generate(
    sample_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"{'PROMPT LENGTH':=^80}")
print(f"Prompt tokens length: {prompt_len}")
print(f"Generated tokens length: {gen_len}")
print(f"Full sequence tokens length: {full_ids.shape[1]}")

print(f"{'FULL SEQUENCE':=^80}")
wrapper = textwrap.TextWrapper(width=80)
print("\n".join(wrapper.fill(line) for line in generation.splitlines()))
print(f"Actual label: {sample_label}")

# Gather residual activations at SAE target layer (full sequence)
print(f"{'RESIDUAL ACTIVATIONS':=^80}")
residual_acts = model.gather_residual_activations(sae_config.layer, full_ids[0])
# residual_acts shape: (n_tokens, d_model)
print(f"Residual activations shape: {residual_acts.shape}")

# Reconstruction & sparsity metrics on full prompt (reported once)
stats = sae.get_reconstruction_stats(residual_acts.float())
print(f"FVU: {stats['fvu']:.2%}")
print(f"L0:  {stats['l0']:.1f}")

# Encode generated-only slice
print(f"{'SAE FEATURE ACTIVATIONS':=^80}")
gen_acts = residual_acts[prompt_len:]
sae_acts_gen = sae.encode(gen_acts.float())
# sae_acts_gen shape: (gen_tokens, n_features)
print(f"SAE activations (gen-only): {sae_acts_gen.shape}")

# Also encode full sequence for later use
sae_acts_full = sae.encode(residual_acts.float())
print(f"SAE activations (full):     {sae_acts_full.shape}")

# Build token strings
all_tokens = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_tokens = all_tokens[prompt_len:]
print(f"All tokens: {len(all_tokens)}, Generated tokens: {len(gen_tokens)}")

# Feature Summary

In [ ]:
from src.utils.visualization import summarize_latents

## Raw Features

In [ ]:
# Generated tokens raw activations.
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)

raw_feature_map = summarize_latents(
    sae_acts_gen, tokens,
    sae_config=sae_config,
    model_name=model_config.model_name,
    top_k=10,
    print_first_n=0,
    full_sae_activations=sae_acts_full,
    all_tokens=all_tokens,
)

In [ ]:
# --- Inspect features using the Feature class ---
example_feature = int(1383)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {raw_feature_map[example_feature]!r}")
print(f"frac_nonzero: {raw_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {raw_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
raw_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

## Denoised Features

In [ ]:
from src.denoiser import Denoiser
from src.configs import DenoisingConfig

# Denoise SAE activations (TF-IDF weighting)
# Denoising happens on the full sequence activations (2-D).
denoising_config = DenoisingConfig(method="continuous_tfidf", params={"threshold": 10.0})
denoiser = Denoiser()
denoised_sae_acts_full = denoiser.denoise(sae_acts_full, denoising_config)

# Extract the generation-only slice.
denoised_sae_acts_gen = denoised_sae_acts_full[prompt_len:, :]
denoised_feature_map = summarize_latents(
    denoised_sae_acts_gen, tokens,
    sae_config=sae_config,
    model_name=model_config.model_name,
    top_k=50,
    print_first_n=0,
    full_sae_activations=sae_acts_full,
    all_tokens=all_tokens,
)

In [ ]:
from src.utils.activations_utils import top_k_features_per_token

dn_per_token_vals, dn_per_token_idxs = top_k_features_per_token(
    denoised_sae_acts_gen, k=10)
print(dn_per_token_vals.min())

In [ ]:
example_feature = int(9)
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {denoised_feature_map[example_feature]!r}")
print(f"frac_nonzero: {denoised_feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {denoised_feature_map[example_feature].top_tokens()}")
print()

# Generation-only view
denoised_feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

In [ ]:
scaling_factors[1228]

In [ ]:
sae_acts_full[:, 1228]

In [ ]:
denoised_sae_acts_full[:, 1228]